# 跨数据源因果信号搜索

在 qlib 原生链路上，用统一、诚实的判定标准（收益+回撤+Sharpe 同时超越 SOXX 且多数年份为正）
系统搜索所有本地可用数据源的因果信号。无收益预测、无拟合参数。


## 1. 预登记计划


In [1]:
import sys, hashlib, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
CARD = ROOT / 'research' / 'cross_source_signal_search_20261010'
plan_bytes = (CARD / 'plan.json').read_bytes()
plan = json.loads(plan_bytes)
print('plan sha256:', hashlib.sha256(plan_bytes).hexdigest())
print('as_of:', plan['as_of']); print('sources:', len(plan['sources'])); print('segments:', plan['segments'])


plan sha256: 3f51f25986bcf37e7c817d8d30d78379a3222d3dd33d134f76ace8d341a7f8d2
as_of: 2026-07-02
sources: 6
segments: {'development': ['2014-01-02', '2022-12-30'], 'reused_diagnostic': ['2023-01-03', '2026-07-02'], 'forward': ['2026-07-03', '2026-10-08']}


## 2. 运行搜索（qlib 原生 + Recorder）


In [2]:
from src.workflow import cross_source_signal_search as workflow
from src.workflow.cross_source_signal_search import write_card_files

summary = workflow.run_cross_source_signal_search(root=ROOT)
write_card_files(summary, card_dir=CARD)
print('decision:', summary['decision'])
print('selected signal:', summary['selected_signal'])
print('stable signals in family:', summary['stable_signals_in_family'])
print('candidate_count:', summary['candidate_count'])
print('recorder_id:', summary['recorder_id'])


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[38553:MainThread](2026-10-09 16:40:53,846) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


2026/10/09 16:40:54 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[38553:MainThread](2026-10-09 16:40:54,235) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[38553:MainThread](2026-10-09 16:40:54,236) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


...

[38553:MainThread](2026-10-09 16:40:54,684) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name cross_source_signal_search_20261010.


[38553:MainThread](2026-10-09 16:40:54,685) INFO - qlib.workflow - [exp.py:258] - Experiment 936440476413716079 starts running ...


[38553:MainThread](2026-10-09 16:40:54,737) INFO - qlib.workflow - [recorder.py:345] - Recorder f1436b48a6534b45ac407adb21ed6db1 starts running under Experiment 936440476413716079 ...


[38553:MainThread](2026-10-09 16:40:54,812) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


decision: no_stable_excess_across_sources
selected signal: None
stable signals in family: []
candidate_count: 36
recorder_id: f1436b48a6534b45ac407adb21ed6db1


## 3. development 信号排名（10bp）


In [3]:
import pandas as pd
comp = pd.read_csv(CARD / 'signal_search.csv')
dev = comp[(comp.window=='development') & (comp.cost_bps==10.0)].sort_values('sharpe_difference', ascending=False)
cols = ['signal','wealth_ratio_vs_soxx','sharpe_difference','max_drawdown_improvement_pct',
        'beats_return','beats_sharpe','beats_drawdown','positive_year_count','year_count','stable_excess']
display(dev[cols].head(12).round(3))
print('development 中稳定超额信号数:', int(dev.stable_excess.sum()))


,signal,wealth_ratio_vs_soxx,sharpe_difference,max_drawdown_improvement_pct,beats_return,beats_sharpe,beats_drawdown,positive_year_count,year_count,stable_excess
0,intraday_realized_vol_d-1_q50,1.708,0.178,-1.212,True,True,True,3,4,False
1,leader_capmkt_reversal_d+1_q35,6.031,0.145,28.278,True,True,False,6,9,False
2,soxl_reversal_d+1_q35,5.235,0.124,32.986,True,True,False,6,9,False
3,leader_breadth_reversal_d+1_q35,4.883,0.112,31.508,True,True,False,6,9,False
4,soxx_reversal_d+1_q35,4.574,0.103,31.365,True,True,False,6,9,False
5,realized_vol_d-1_q50,2.192,0.098,3.769,True,True,False,5,9,False
6,leader_breadth_reversal_d+1_q50,4.084,0.090,30.151,True,True,False,6,9,False
7,leader_capmkt_reversal_d+1_q50,3.738,0.077,32.804,True,True,False,6,9,False
8,soxl_reversal_d+1_q50,3.370,0.061,31.652,True,True,False,6,9,False
9,trend_d+1_q35,2.224,0.009,15.565,True,True,False,5,9,False


development 中稳定超额信号数: 0


## 4. 各窗口全部结果汇总


In [4]:
pivot = comp.pivot_table(index='signal', columns=['window','cost_bps'],
                         values='sharpe_difference', aggfunc='first')
print('任一窗口/cost 达成稳定超额:', int(comp.stable_excess.sum()))
display(pivot.round(3).head(20))


任一窗口/cost 达成稳定超额: 0


window                          development        forward       \
cost_bps                               10.0   25.0    10.0 25.0   
signal                                                            
intraday_close_location_d+1_q35      -0.283 -0.495     NaN  NaN   
intraday_close_location_d+1_q50      -0.401 -0.651     NaN  NaN   
intraday_close_location_d-1_q35      -0.304 -0.535     NaN  NaN   
intraday_close_location_d-1_q50      -0.223 -0.456     NaN  NaN   
intraday_early_momentum_d+1_q35      -0.145 -0.346     NaN  NaN   
intraday_early_momentum_d+1_q50      -0.063 -0.271     NaN  NaN   
intraday_early_momentum_d-1_q35      -0.468 -0.713     NaN  NaN   
intraday_early_momentum_d-1_q50      -0.535 -0.761     NaN  NaN   
intraday_realized_vol_d+1_q35        -0.222 -0.279     NaN  NaN   
intraday_realized_vol_d+1_q50        -0.351 -0.421     NaN  NaN   
intraday_realized_vol_d-1_q35        -0.013 -0.141     NaN  NaN   
intraday_realized_vol_d-1_q50         0.178  0.042     NaN  NaN   
leader_breadth_reversal_d+1_q35       0.112 -0.123     NaN  NaN   
leader_breadth_reversal_d+1_q50       0.090 -0.183     NaN  NaN   
leader_breadth_reversal_d-1_q35      -0.745 -1.059     NaN  NaN   
leader_breadth_reversal_d-1_q50      -0.700 -0.999     NaN  NaN   
leader_capmkt_reversal_d+1_q35        0.145 -0.099     NaN  NaN   
leader_capmkt_reversal_d+1_q50        0.077 -0.195     NaN  NaN   
leader_capmkt_reversal_d-1_q35       -0.807 -1.114     NaN  NaN   
leader_capmkt_reversal_d-1_q50       -0.695 -0.996     NaN  NaN   

window                          reused_diagnostic         
cost_bps                                     10.0   25.0  
signal                                                    
intraday_close_location_d+1_q35            -0.391 -0.563  
intraday_close_location_d+1_q50            -0.459 -0.676  
intraday_close_location_d-1_q35            -0.364 -0.591  
intraday_close_location_d-1_q50            -0.377 -0.592  
intraday_early_momentum_d+1_q35            -0.483 -0.660  
intraday_early_momentum_d+1_q50            -0.798 -1.016  
intraday_early_momentum_d-1_q35            -0.254 -0.481  
intraday_early_momentum_d-1_q50             0.010 -0.227  
intraday_realized_vol_d+1_q35              -0.209 -0.312  
intraday_realized_vol_d+1_q50              -0.299 -0.396  
intraday_realized_vol_d-1_q35              -0.481 -0.644  
intraday_realized_vol_d-1_q50              -0.329 -0.469  
leader_breadth_reversal_d+1_q35            -0.209 -0.388  
leader_breadth_reversal_d+1_q50            -0.231 -0.459  
leader_breadth_reversal_d-1_q35            -0.521 -0.776  
leader_breadth_reversal_d-1_q50            -0.506 -0.740  
leader_capmkt_reversal_d+1_q35             -0.453 -0.648  
leader_capmkt_reversal_d+1_q50             -0.217 -0.448  
leader_capmkt_reversal_d-1_q35             -0.328 -0.572  
leader_capmkt_reversal_d-1_q50             -0.622 -0.858

## 5. 中文结论


In [5]:
print('候选族:', summary['candidate_count'], '条；数据源:', summary['signal_sources'])
print('稳定超额信号:', summary['stable_signals_in_family'] or '无')
print('决定:', summary['decision'], '| promotion_eligible:', summary['promotion_eligible'])
print('qlib Recorder:', summary['recorder_id'])


候选族: 36 条；数据源: ['soxl_daily', 'soxl_intraday_t1', 'soxx_constituents', 'soxx_daily']
稳定超额信号: 无
决定: no_stable_excess_across_sources | promotion_eligible: False
qlib Recorder: f1436b48a6534b45ac407adb21ed6db1
